In [1]:
import os
import json
import numpy as np
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import models
import torchvision.transforms as T

import albumentations as A
from albumentations.pytorch import ToTensorV2

In [2]:
class ZeroWasteDataset(Dataset):
    def __init__(self, root_dir, split='train', transform=None):
        """
        Args:
            root_dir (str): Root directory containing the splits_final_deblurred folder.
            split (str): One of 'train', 'val', or 'test'.
            transform: Albumentations transform (applied jointly on image and mask)
        """
        self.split_dir = os.path.join(root_dir, split)
        self.data_dir = os.path.join(self.split_dir, 'data')
        self.mask_dir = os.path.join(self.split_dir, 'sem_seg')
        self.transform = transform

        # Read labels.json (if needed)
        labels_path = os.path.join(self.split_dir, 'labels.json')
        with open(labels_path, 'r') as f:
            self.labels_data = json.load(f)
        
        self.image_files = sorted(os.listdir(self.data_dir))
        self.mask_files = sorted(os.listdir(self.mask_dir))

    def __len__(self):
        return len(self.image_files)

    def __getitem__(self, idx):
        # Load image and mask
        img_path = os.path.join(self.data_dir, self.image_files[idx])
        mask_path = os.path.join(self.mask_dir, self.mask_files[idx])
        
        image = np.array(Image.open(img_path).convert("RGB"))
        mask = np.array(Image.open(mask_path))
        # Subtract 1 so that object classes (1,2,3,4) become (0,1,2,3)
        # и фон (0) становится -1, который будет игнорироваться.
        if self.transform is not None:
            augmented = self.transform(image=image, mask=mask)
            image = augmented['image']
            mask = augmented['mask'].long() - 1
        else:
            image = T.ToTensor()(image)
            mask = torch.as_tensor(mask, dtype=torch.long) - 1
            
        return image, mask

In [3]:
class CustomDeepLabHead(nn.Module):
    def __init__(self, in_channels, num_classes, dropout=0.5):
        super(CustomDeepLabHead, self).__init__()
        self.conv1 = nn.Conv2d(in_channels, 256, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(256)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(dropout)
        self.conv2 = nn.Conv2d(256, num_classes, kernel_size=1)
    
    def forward(self, x):
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu(x)
        x = self.dropout(x)
        x = self.conv2(x)
        return x

In [4]:
def train_model(model, dataloaders, criterion, optimizer, scheduler, device, num_epochs=25):
    best_loss = float('inf')
    for epoch in range(num_epochs):
        print(f"Epoch {epoch+1}/{num_epochs}")
        model.train()
        running_loss = 0.0
        
        # Training phase
        for images, masks in dataloaders['train']:
            images = images.to(device)
            masks = masks.to(device)
            
            optimizer.zero_grad()
            outputs = model(images)['out']
            loss = criterion(outputs, masks)
            loss.backward()
            # Gradient clipping to avoid exploding gradients
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            running_loss += loss.item() * images.size(0)
            
        epoch_loss = running_loss / len(dataloaders['train'].dataset)
        print(f"Train Loss: {epoch_loss:.4f}")
        
        # Validation phase
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for images, masks in dataloaders['val']:
                images = images.to(device)
                masks = masks.to(device)
                outputs = model(images)['out']
                loss = criterion(outputs, masks)
                val_loss += loss.item() * images.size(0)
        epoch_val_loss = val_loss / len(dataloaders['val'].dataset)
        print(f"Val Loss: {epoch_val_loss:.4f}")
        
        # Update scheduler based on validation loss
        scheduler.step(epoch_val_loss)
        
        if epoch_val_loss < best_loss:
            best_loss = epoch_val_loss
            torch.save(model.state_dict(), 'best_model.pth')
            print("Best model saved")
    return model

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

data_root = '/home/dtsarev/master_of_cv/sem3/DL_project/data/splits_final_deblurred/'

# Augmentation for training data
train_transform = A.Compose([
    A.RandomResizedCrop(height=512, width=512, scale=(0.8, 1.0), ratio=(0.75, 1.33), p=1.0),
    A.HorizontalFlip(p=0.5),
    A.Rotate(limit=15, p=0.5),
    A.RandomBrightnessContrast(p=0.5),
    A.Normalize(mean=(0.485, 0.456, 0.406),
                std=(0.229, 0.224, 0.225)),
    ToTensorV2()
])
# For validation
val_transform = A.Compose([
    A.Resize(height=512, width=512),
    A.Normalize(mean=(0.485, 0.456, 0.406),
                std=(0.229, 0.224, 0.225)),
    ToTensorV2()
])

# Create datasets and dataloaders
train_dataset = ZeroWasteDataset(root_dir=data_root, split='train', transform=train_transform)
val_dataset = ZeroWasteDataset(root_dir=data_root, split='val', transform=val_transform)

train_loader = DataLoader(train_dataset, batch_size=5, shuffle=True, num_workers=4)
val_loader = DataLoader(val_dataset, batch_size=5, shuffle=False, num_workers=4)
dataloaders = {'train': train_loader, 'val': val_loader}

# Using 4 classes (ignore background)
num_classes = 4

model = models.segmentation.deeplabv3_resnet101(pretrained=True)
# Chenge the classifier by the custom one with dropout
model.classifier = CustomDeepLabHead(2048, num_classes, dropout=0.5)
model.to(device)

# Class weights (after substracting 1)
# rigid_plastic: 1130, cardboard: 11941, metal: 259, soft_plastic: 4672.
weight_rigid = 11941 / 1130  # ~10.57
weight_cardboard = 11941 / 11941  # 1.0
weight_metal = 11941 / 259  # ~46.08
weight_soft = 11941 / 4672  # ~2.56
class_weights = torch.tensor([weight_rigid, weight_cardboard, weight_metal, weight_soft],
                             dtype=torch.float).to(device)

# Using ignore_index=-1 to ignore background
criterion = nn.CrossEntropyLoss(weight=class_weights, ignore_index=-1)

# L2 regularization
optimizer = optim.Adam(model.parameters(), lr=1e-4, weight_decay=1e-5)

# Scheduler for decreasing LR plateau
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3)

num_epochs = 25
trained_model = train_model(model, dataloaders, criterion, optimizer, scheduler, device, num_epochs=num_epochs)

print("Training completed!")

Using device: cuda


/home/dtsarev/anaconda3/lib/python3.11/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/dtsarev/anaconda3/lib/python3.11/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=DeepLabV3_ResNet101_Weights.COCO_WITH_VOC_LABELS_V1`. You can also use `weights=DeepLabV3_ResNet101_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Epoch 1/25
Train Loss: 0.7786
Val Loss: 0.7897
Best model saved
Epoch 2/25
Train Loss: 0.5604
Val Loss: 0.9096
Epoch 3/25
Train Loss: 0.4883
Val Loss: 0.9399
Epoch 4/25
Train Loss: 0.4172
Val Loss: 0.7859
Best model saved
Epoch 5/25
Train Loss: 0.3689
Val Loss: 1.0342
Epoch 6/25
Train Loss: 0.3234
Val Loss: 1.3296
Epoch 7/25
